# chugchug — Next-Generation Progress Bars

Beautiful gradient progress bars that work everywhere — terminals, notebooks, CI/CD.

**Zero dependencies. Event-driven. Multiprocessing-safe.**

In [ ]:
import time
import math
import random
from chugchug import chug, Chug

## 1. Basic usage — one line, beautiful by default

In [ ]:
for _ in chug(range(200), desc="Processing"):
    time.sleep(0.01)

## 2. All gradient presets

14 built-in gradients including multi-stop presets like `rainbow`, `aurora`, and `candy`.

In [ ]:
gradients = [
    "ocean", "fire", "forest", "purple", "cyber", "mono",
    "rainbow", "heatmap", "candy", "neon", "aurora", "sunset", "matrix", "ice",
]
for g in gradients:
    for _ in chug(range(80), desc=f"{g:8s}", gradient=g):
        time.sleep(0.003)

## 3. ML Training — metrics displayed inline

Track loss, accuracy, learning rate — all rendered live inside the bar.

In [ ]:
b = Chug(total=80, desc="Training", gradient="fire", unit="step")
for step in range(80):
    loss = 2.5 * math.exp(-step / 20) + random.gauss(0, 0.05)
    acc = min(0.99, 0.5 + 0.5 * (1 - math.exp(-step / 15)))
    lr = 0.001 * (0.95 ** step)
    b.set_metrics(loss=f"{loss:.4f}", acc=f"{acc:.1%}", lr=f"{lr:.1e}")
    b.update()
    time.sleep(0.04)
b.close()

## 4. Smart ETA — adapts to speed changes

Watch the ETA adapt in real time as processing speed varies.

In [ ]:
b = Chug(total=120, desc="Variable speed", gradient="aurora")
for i in range(120):
    delay = 0.01 + 0.04 * math.sin(i / 10) ** 2
    time.sleep(delay)
    b.update()
b.close()

## 5. Unknown total — spinner mode

When you don't know the total, chugchug shows count + elapsed + rate.

In [ ]:
b = Chug(desc="Streaming", unit="rec", gradient="cyber")
for i in range(100):
    b.update()
    time.sleep(0.02)
b.close()

## 6. Unit scaling — human-readable large numbers

In [ ]:
b = Chug(total=1_000_000, desc="Tokens", unit="tok", unit_scale=True, gradient="neon")
for _ in range(100):
    b.update(10_000)
    time.sleep(0.03)
b.close()

## 7. Custom gradient — register your own colors

In [ ]:
from chugchug import register_gradient, register_multi_gradient

# Simple 2-stop
register_gradient("coral", (255, 94, 77), (255, 195, 113))
for _ in chug(range(80), desc="Coral", gradient="coral"):
    time.sleep(0.008)

# Multi-stop
register_multi_gradient("vaporwave", [
    (255, 0, 128), (128, 0, 255), (0, 200, 255),
    (128, 255, 128), (255, 255, 0), (255, 0, 128),
])
for _ in chug(range(80), desc="Vaporwave", gradient="vaporwave"):
    time.sleep(0.008)

## 8. Smart generators — wraps map/enumerate/genexpr

tqdm shows `0it [00:00, ?it/s]` for these. chugchug extracts the total automatically.

In [ ]:
data = list(range(200))

# map() — chugchug detects total=200
for _ in chug(map(str, data), desc="map()"):
    time.sleep(0.004)

# enumerate() — also works
for _ in chug(enumerate(data), desc="enumerate()"):
    time.sleep(0.004)

# generator expression — even this works
for _ in chug((x**2 for x in data), desc="genexpr"):
    time.sleep(0.004)

## 9. tqdm drop-in replacement

Swap `from tqdm import tqdm` → `from chugchug.compat import tqdm`. Done.

In [ ]:
from chugchug.compat import tqdm, trange

for _ in tqdm(range(100), desc="tqdm compat"):
    time.sleep(0.01)

## 10. Crash context — know exactly where it died

In [ ]:
try:
    with Chug(range(100), desc="Processing", gradient="fire") as b:
        for i in b:
            if i == 42:
                raise ValueError("data corrupted at row 42")
            time.sleep(0.01)
except ValueError as e:
    print(f"Caught: {e}")